# ModelSentry POC validation

This notebook exercises the four SDK modules end-to-end and validates the three
technical assumptions behind ModelSentry:

1. **Local statistical profiling works** — a `Profile` object with correct stats is produced from raw features and predictions.
2. **Framework-agnostic instrumentation works** — the same `@ms.monitor()` decorator wraps both a scikit-learn model and a pure NumPy function.
3. **Deliberately introduced drift is detectable** — `detect_drift()` flags shifted features as `warning` or `critical`.

Run all cells top to bottom. The final cell prints PASS/FAIL for each assumption.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

import modelsentry as ms
from modelsentry.monitor import flush, get_latest_profile, shutdown
from modelsentry.profiler import profile
from modelsentry.drift import detect_drift

RNG = np.random.default_rng(42)
results: dict[str, bool] = {}

## Synthetic churn dataset

Five features (3 numeric, 2 categorical) and a binary churn label. We generate
two populations: a **baseline** (representative of training-time data) and a
**drifted** version with `age` and `income` shifted to simulate a population
shift seen in production.

In [ ]:
COUNTRIES = ["US", "CA", "UK", "DE", "FR"]
TIERS = ["free", "pro", "enterprise"]


def generate_churn_data(n, age_loc, income_loc, income_scale, seed):
    rng = np.random.default_rng(seed)
    age = rng.normal(age_loc, 10, size=n).clip(18, 90)
    income = rng.normal(income_loc, income_scale, size=n).clip(0, None)
    tenure = rng.uniform(0, 60, size=n)
    country = rng.choice(COUNTRIES, size=n)
    tier = rng.choice(TIERS, size=n, p=[0.6, 0.3, 0.1])
    df = pd.DataFrame({
        "age": age, "income": income, "tenure": tenure,
        "country": country, "tier": tier,
    })
    score = (
        0.05 * (age - 40)
        + -0.00002 * (income - 50000)
        + -0.04 * (tenure - 30)
        + rng.normal(0, 1, size=n)
    )
    churn = (score > 0).astype(int)
    return df, churn


baseline_df, baseline_y = generate_churn_data(
    n=1000, age_loc=40, income_loc=50000, income_scale=15000, seed=1
)
drifted_df, _ = generate_churn_data(
    n=1000, age_loc=55, income_loc=65000, income_scale=25000, seed=2
)
print("baseline head:")
print(baseline_df.head())
print(chr(10) + "baseline describe:")
print(baseline_df.describe())

## Train a scikit-learn churn model

A simple `LogisticRegression` over numeric features and one-hot-encoded
categoricals. The `predict` function returns `"churn"` / `"stay"` string
labels so the SDK profiles predictions as a classification task.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(baseline_df[["country", "tier"]])


def _to_matrix(df: pd.DataFrame) -> np.ndarray:
    cat = encoder.transform(df[["country", "tier"]])
    num = df[["age", "income", "tenure"]].to_numpy()
    return np.hstack([num, cat])


model = LogisticRegression(max_iter=1000)
model.fit(_to_matrix(baseline_df), baseline_y)


def predict(features: pd.DataFrame) -> np.ndarray:
    raw = model.predict(_to_matrix(features))
    return np.where(raw == 1, "churn", "stay")


print(f"trained accuracy: {model.score(_to_matrix(baseline_df), baseline_y):.3f}")

## Assumption 1 — Local statistical profiling

Wrap `predict` with `@ms.monitor()`, run the baseline through it in 10 batches
of 100 rows, and confirm that the produced `Profile` has the expected
structure: 5 features, correct dtypes, classification task type, 1000 rows.

In [ ]:
ms.init(api_key="poc-test", model_id="churn-v3", profile_window=10,
        profile_handler=lambda p, mid: None)


@ms.monitor()
def monitored_predict(features):
    return predict(features)


for i in range(10):
    chunk = baseline_df.iloc[i * 100:(i + 1) * 100]
    monitored_predict(chunk)
flush()

profile_a1 = get_latest_profile()
print(f"n_rows: {profile_a1.n_rows}")
print(f"features: {list(profile_a1.feature_profiles)}")
print(f"task:    {profile_a1.prediction_profile.task_type}")
print(f"age stats: mean={profile_a1.feature_profiles['age'].numeric_stats.mean:.2f} "
      f"std={profile_a1.feature_profiles['age'].numeric_stats.std:.2f}")
print(f"country categories: {list(profile_a1.feature_profiles['country'].value_counts)}")

In [ ]:
expected_features = {"age", "income", "tenure", "country", "tier"}
a1_pass = (
    profile_a1 is not None
    and profile_a1.n_rows == 1000
    and set(profile_a1.feature_profiles) == expected_features
    and profile_a1.feature_profiles["age"].dtype == "numeric"
    and profile_a1.feature_profiles["country"].dtype == "categorical"
    and profile_a1.prediction_profile.task_type == "classification"
)
results["A1: local statistical profiling"] = a1_pass
print("Assumption 1: " + ("PASS" if a1_pass else "FAIL"))
shutdown()

## Assumption 2 — Framework-agnostic instrumentation

The same `@ms.monitor()` decorator wraps a hand-rolled NumPy classifier with no
sklearn or framework imports inside the predict function. The Profile produced
should have the same structure as the sklearn case, proving the decorator
intercepts at the data boundary, not the model.

In [ ]:
ms.init(api_key="poc-test", model_id="churn-numpy", profile_window=1,
        profile_handler=lambda p, mid: None)

weights = RNG.normal(size=3)
bias = RNG.normal()


@ms.monitor()
def numpy_predict(features):
    x = features[["age", "income", "tenure"]].to_numpy()
    logits = x @ weights + bias
    return np.where(logits > 0, "churn", "stay")


numpy_predict(baseline_df)
flush()
profile_a2 = get_latest_profile()
print(f"n_rows: {profile_a2.n_rows}")
print(f"features: {list(profile_a2.feature_profiles)}")
print(f"task:    {profile_a2.prediction_profile.task_type}")

In [ ]:
a2_pass = (
    profile_a2 is not None
    and profile_a2.n_rows == 1000
    and "age" in profile_a2.feature_profiles
    and profile_a2.prediction_profile.task_type == "classification"
)
results["A2: framework-agnostic instrumentation"] = a2_pass
print("Assumption 2: " + ("PASS" if a2_pass else "FAIL"))
shutdown()

## Assumption 3 — Drift is detectable

Profile the baseline, profile the drifted population using the **same bin
edges** (so PSI is comparable), then run `detect_drift`. We expect:

- `age` and `income` flagged `warning` or `critical` — they shifted
- `tenure`, `country`, `tier` flagged `stable` — they didn't

In [ ]:
# Direct profile() calls — clearer than going through the buffered monitor.
preds_baseline = predict(baseline_df)
preds_drifted = predict(drifted_df)

baseline_profile = profile(baseline_df, preds_baseline)

# Reuse baseline edges so the two profiles' histograms align bin-for-bin.
edges = {
    name: fp.distribution.bin_edges
    for name, fp in baseline_profile.feature_profiles.items()
    if fp.distribution is not None
}
current_profile = profile(drifted_df, preds_drifted, baseline_edges=edges)

report = detect_drift(baseline_profile, current_profile)
print(f"Overall severity: {report.overall_severity}")
print()
for name, r in report.feature_results.items():
    ks = f"ks_p={r.ks_p_value:.4f}" if r.ks_p_value is not None else "ks_p=N/A"
    print(f"  {name:10s} severity={r.severity:8s} psi={r.psi:7.4f}  {ks}")

In [ ]:
a3_pass = (
    report.overall_severity in ("warning", "critical")
    and report.feature_results["age"].severity in ("warning", "critical")
    and report.feature_results["income"].severity in ("warning", "critical")
    and report.feature_results["tenure"].severity == "stable"
    and report.feature_results["country"].severity == "stable"
    and report.feature_results["tier"].severity == "stable"
)
results["A3: deliberate drift is detectable"] = a3_pass
print("Assumption 3: " + ("PASS" if a3_pass else "FAIL"))

## Summary

In [ ]:
print("=" * 60)
print("ModelSentry POC validation — final results")
print("=" * 60)
for name, ok in results.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print("=" * 60)
all_pass = all(results.values())
print()
print('All assumptions PASSED — POC complete.' if all_pass else 'Some assumptions FAILED — see above.')
assert all_pass, 'POC validation failed'